In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import glob
import shap
import json
import networkx as nx
import seaborn as sns
import lingam
import warnings
import requests

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler,
    MinMaxScaler,
)

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from typing import Dict
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score
from scipy.stats import pearsonr
from statsmodels.tsa.stattools import ccf
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.api import VAR
from scipy import stats
from dotenv import load_dotenv
from collections import defaultdict
from IPython.display import display, Markdown
from datetime import datetime

from sklearn.metrics import (
    roc_auc_score, 
    roc_curve, 
    confusion_matrix, 
    precision_score, 
    recall_score, 
    accuracy_score,
    classification_report
)


load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

warnings.filterwarnings('ignore')
print("✅ Imports loaded successfully!")

✅ Imports loaded successfully!


<br> <br> <br>

### Read the the .csv file

In [2]:
machine_number = 72

In [3]:
lag_features_path = f"../../data/azure_pm/lag_features/machine_{machine_number}_lag_features.csv"
df_machine = pd.read_csv(lag_features_path)
print(f"Loaded shape: {df_machine.shape}")
df_machine.head()

Loaded shape: (8741, 65)


,datetime,volt,rotate,pressure,vibration,errorID,comp,failure,target,volt_lag_1h,...,error_count_6h,error_count_24h,maint_count_6h,maint_count_24h,hour,day_of_week,is_weekend,is_working_hours,hours_since_maint,hours_since_error
0,2015-01-01 06:00:00,0.463039,0.562292,0.491340,0.537406,0,0,0,0,0.000000,...,0.0,0.0,0.0,0.0,6,3,0,0,0,0
1,2015-01-01 07:00:00,0.618236,0.478765,0.339592,0.342058,0,0,0,0,0.463039,...,0.0,0.0,0.0,0.0,7,3,0,0,1,1
2,2015-01-01 08:00:00,0.554276,0.596626,0.535207,0.616585,0,0,0,0,0.618236,...,0.0,0.0,0.0,0.0,8,3,0,1,2,2
3,2015-01-01 09:00:00,0.335221,0.339212,0.190289,0.489716,0,0,0,0,0.554276,...,0.0,0.0,0.0,0.0,9,3,0,1,3,3
4,2015-01-01 10:00:00,0.639455,0.354593,0.306272,0.452145,0,0,0,0,0.335221,...,0.0,0.0,0.0,0.0,10,3,0,1,4,4


In [4]:
df_machine.head()

,datetime,volt,rotate,pressure,vibration,errorID,comp,failure,target,volt_lag_1h,...,error_count_6h,error_count_24h,maint_count_6h,maint_count_24h,hour,day_of_week,is_weekend,is_working_hours,hours_since_maint,hours_since_error
0,2015-01-01 06:00:00,0.463039,0.562292,0.491340,0.537406,0,0,0,0,0.000000,...,0.0,0.0,0.0,0.0,6,3,0,0,0,0
1,2015-01-01 07:00:00,0.618236,0.478765,0.339592,0.342058,0,0,0,0,0.463039,...,0.0,0.0,0.0,0.0,7,3,0,0,1,1
2,2015-01-01 08:00:00,0.554276,0.596626,0.535207,0.616585,0,0,0,0,0.618236,...,0.0,0.0,0.0,0.0,8,3,0,1,2,2
3,2015-01-01 09:00:00,0.335221,0.339212,0.190289,0.489716,0,0,0,0,0.554276,...,0.0,0.0,0.0,0.0,9,3,0,1,3,3
4,2015-01-01 10:00:00,0.639455,0.354593,0.306272,0.452145,0,0,0,0,0.335221,...,0.0,0.0,0.0,0.0,10,3,0,1,4,4


<br> <br>

### Create/Read Causal graph file

In [5]:
def create_causal_network(df, features, regularize=True, noise_level=1e-4):
    X_selected = df[features].values

    # Initialize the VARLiNGAM model
    model = lingam.VARLiNGAM()

    # Attempt to fit the model; add noise if a LinAlgError occurs
    try:
        model.fit(X_selected)
    except np.linalg.LinAlgError as e:
        print("LinAlgError encountered during model fitting:", e)
        if regularize:
            print(f"Applying regularization: adding noise (std={noise_level}) to the data and trying again.")
            X_selected += np.random.normal(0, noise_level, X_selected.shape)
            model.fit(X_selected)
        else:
            raise e

    # Retrieve the adjacency matrices (one per lag)
    adjacency_matrices = model.adjacency_matrices_

    # Build a directed graph and create a list of dictionaries for each non-zero edge.
    G = nx.DiGraph()
    G.add_nodes_from(features)
    n_features = len(features)
    structured_adjacency = []

    # Loop through each adjacency matrix (one per lag)
    for lag_index, adj_matrix in enumerate(adjacency_matrices):
        for i in range(n_features):
            for j in range(n_features):
                weight = adj_matrix[i, j]
                if weight != 0:
                    edge_dict = {
                        "source_feature": features[i],
                        "target_feature": features[j],
                        "effect_strength": weight,
                    }
                    structured_adjacency.append(edge_dict)
                    G.add_edge(features[i], features[j], weight=weight)

    # Plot the graph using a spring layout for clarity
    pos = nx.spring_layout(G, seed=42)  # seed for reproducibility
    fig, ax = plt.subplots(figsize=(12, 8))
    nx.draw(
        G,
        pos,
        with_labels=True,
        node_color="lightblue",
        node_size=1500,
        arrowstyle="->",
        arrowsize=20,
        edge_color="gray",
        font_size=10,
        ax=ax,
    )

    # Display the edge weights formatted to two decimal places
    edge_labels = nx.get_edge_attributes(G, "weight")
    edge_labels = {edge: f"{weight:.2f}" for edge, weight in edge_labels.items()}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color="red", ax=ax)

    ax.set_title("Causal Network Graph from VARLiNGAM")
    ax.axis("off")
    # plt.show()
    plt.close()

    return structured_adjacency, fig



causal_path = f"../../causality_graph/azure_pm/machine_{machine_number}/structured_adjacency.json"

if os.path.exists(causal_path):
    with open(causal_path, "r") as f:
        structured_adjacency = json.load(f)
else:
    os.makedirs(os.path.dirname(causal_path), exist_ok=True)
    # Use all columns except non-feature columns for causal discovery
    feature_cols = [col for col in df_machine.columns if col not in ["datetime", "failure", "target"]]
    structured_adjacency, fig = create_causal_network(df_machine, feature_cols)
    with open(causal_path, "w") as f:
        json.dump(structured_adjacency, f, indent=2)

print(f"Loaded {len(structured_adjacency)} edges from structured_adjacency.json")

# print(structured_adjacency[0:2])

Loaded 980 edges from structured_adjacency.json


In [ ]:
# df_machine_98.drop(columns=["datetime", "failure", "errorID"])

<br> <br>

## Read ML models

In [6]:
model_dir = f"../../model/azure_pm/machine_{machine_number}"
pkl_files = glob.glob(os.path.join(model_dir, "*.pkl"))

models = {}
for file_path in pkl_files:
    model_name = os.path.splitext(os.path.basename(file_path))[0]
    with open(file_path, "rb") as f:
        models[model_name] = pickle.load(f)

print(f"Loaded {len(models)} models: {list(models.keys())}")

my_models = {
    "catboost_model": 'CatBoost', 
    "lightgbm_model": 'LightGBM', 
    "randomforest_model": 'RandomForest', 
    "xgboost_model": 'XGBoost'
}

# model = models["XGBoost"]
model = models[my_models["catboost_model"]]
# model = models[my_models["lightgbm_model"]]
# model = models[my_models["randomforest_model"]]
# model = models[my_models["xgboost_model"]]
# model



Loaded 4 models: ['CatBoost', 'LightGBM', 'RandomForest', 'XGBoost']


<br> <br>

## Generate a random row

In [ ]:
def get_random_nonzero_target_row(df):
    filtered = df[df["target"] != 0]
    # filtered = df[(df["target"] != 0) & (df["failure"]==0)]
    if filtered.empty:
        return None
    return filtered.sample(n=1, random_state=np.random.randint(0, 10000)).iloc[0]


# Example usage:
random_row = get_random_nonzero_target_row(df_machine)
random_row_full = random_row.copy()
random_row.drop(["datetime", "failure", "target"], inplace=True)

# random_row.apply(pd.to_numeric)
display(random_row.to_frame().T)

,volt,rotate,pressure,vibration,errorID,comp,volt_lag_1h,volt_lag_6h,volt_lag_12h,volt_lag_24h,...,error_count_6h,error_count_24h,maint_count_6h,maint_count_24h,hour,day_of_week,is_weekend,is_working_hours,hours_since_maint,hours_since_error
5914,0.588993,0.722968,0.723416,0.513045,0,0,0.557697,0.336036,0.75806,0.601954,...,0.0,1.0,0.0,0.0,13,4,0,1,343,7


In [16]:
df_machine[(df_machine["target"] != 0) | (df_machine["failure"] != 0)]     [["datetime", "failure", "target"]]


,datetime,failure,target
1944,2015-03-23 06:00:00,0,1
1945,2015-03-23 07:00:00,0,1
1946,2015-03-23 08:00:00,0,1
1947,2015-03-23 09:00:00,0,1
1948,2015-03-23 10:00:00,0,1
1949,2015-03-23 11:00:00,0,1
1950,2015-03-23 12:00:00,0,1
1951,2015-03-23 13:00:00,0,1
1952,2015-03-23 14:00:00,0,1
1953,2015-03-23 15:00:00,0,1


In [14]:
df_machine[(df_machine["failure"] != 0)][["datetime", "failure", "target"]]

,datetime,failure,target
1968,2015-03-24 06:00:00,1,0
5931,2015-09-05 06:00:00,1,0


In [ ]:
# Check the failure and target variables 24 hours before a failure occures
df_machine_original = pd.read_csv(f"../../data/azure_pm/machines/machine_{machine_number}.csv")

display(df_machine_original.iloc[random_row_full.to_frame().T.index.values[0]-24:random_row_full.to_frame().T.index.values[0]+1][["datetime", "failure"]])
display(df_machine.iloc[random_row_full.to_frame().T.index.values[0]-24:random_row_full.to_frame().T.index.values[0]+1][["datetime","failure", "target"]])

<br> <br> <br>

## Find Uncertain Rows

In [ ]:
# Find rows where the model prediction has low confidence (uncertain) and target != 0

# Get model and feature names
clf = model['model']
feature_names = model['feature_names']

# Prepare input data (exclude non-feature columns if present)
X = df_machine[feature_names]
y = df_machine['target']

# Get prediction probabilities
probs = clf.predict_proba(X)
max_probs = probs.max(axis=1)

# Define uncertainty threshold (e.g., max prob < 0.5 means uncertain)
uncertainty_threshold = 0.5
uncertain_mask = (max_probs < uncertainty_threshold) # & (y != 0)

uncertain_rows = df_machine[uncertain_mask]
print(f"Found {len(uncertain_rows)} uncertain rows with nonzero target.")


In [ ]:
# uncertain_rows
# random_row = uncertain_rows.iloc[0]

<br> <br> 

## Generate top-n rows using SHAP

In [ ]:
def get_top_n_shap_features(random_row, model_dict, top_n=10):
    model = model_dict['model']
    feature_names = model_dict['feature_names']

    # Prepare the row as a DataFrame for SHAP and ensure numeric types
    X_row = random_row.to_frame().T[feature_names]
    X_row = X_row.apply(pd.to_numeric, errors='coerce')

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_row)

    if isinstance(shap_values, list):
        shap_abs = np.sum([np.abs(sv).reshape(-1) for sv in shap_values], axis=0)
    else:
        shap_abs = np.abs(shap_values).reshape(-1)

    min_len = min(len(feature_names), len(shap_abs))
    df_shap = pd.DataFrame({
        'feature_name': feature_names[:min_len],
        'feature_importance': shap_abs[:min_len]
    }).sort_values('feature_importance', ascending=False).head(top_n).reset_index(drop=True)

    return df_shap


df_shap = get_top_n_shap_features(random_row=random_row, model_dict=model, top_n=10)
display(df_shap)


# Visualize SHAP values from df_shap as a horizontal bar plot
plt.figure(figsize=(8, 6))
plt.barh(df_shap['feature_name'][::-1], df_shap['feature_importance'][::-1], color='skyblue')
plt.xlabel('SHAP Value (absolute)')
plt.title('Top SHAP Feature Importances')
plt.show()

<br> <br> <br>

### See the hierarchy

In [ ]:
def format_taxonomy(taxonomy_json):
    """Formats a hierarchical taxonomy into a tree-like string."""
    if isinstance(taxonomy_json, str):
        taxonomy_json = json.loads(taxonomy_json)  # Parse JSON string if needed

    # Build tree structure
    tree = defaultdict(list)
    for feature, (_, parent) in taxonomy_json.items():
        tree[parent].append(feature)

    def display_tree(parent="Root", level=0):
        """Recursively formats the tree structure with icons and connectors."""
        if parent not in tree:
            return ""

        result = ""
        for idx, child in enumerate(tree[parent]):
            is_parent = child in tree  # Check if child has further children
            icon = "📂" if is_parent else "📄"  # Folder for parents, file for leaf nodes
            connector = "└── " if idx == len(tree[parent]) - 1 else "├── "  # Tree connectors
            result += "  " * level + connector + f"{icon} {child}\n"
            result += display_tree(child, level + 1)  # Recursive call for children

        return result

    return f"\n📌 **Taxonomy Structure:**\n\n{display_tree('Root')}".strip()

def get_taxonomy_paths(taxonomy_json, node_names):
    """Returns hierarchical paths for given nodes in a taxonomy."""
    if isinstance(taxonomy_json, str):
        taxonomy_json = json.loads(taxonomy_json)  # Parse JSON if it's a string

    # Reverse lookup: Map each child to its parent
    child_to_parent = {child: parent for child, (_, parent) in taxonomy_json.items()}

    def get_path(node):
        """Recursively constructs the hierarchy path for a node."""
        path = [node]
        while node in child_to_parent and child_to_parent[node] != "Root":
            node = child_to_parent[node]
            path.append(node)
        return " --> ".join(reversed(path))  # Reverse to get correct hierarchy order
    return {node: get_path(node) for node in node_names}


with open('../../taxonomy/deepseek.json', 'r') as file:
    deepseek_taxonomy = json.load(file)

with open('../../taxonomy/gemma2.json', 'r') as file:
    gemma2_taxonomy = json.load(file)

with open('../../taxonomy/llama3.json', 'r') as file:
    llama3_taxonomy = json.load(file)

with open('../../taxonomy/gpt4o.json', 'r') as file:
    gpt4o_taxonomy = json.load(file)

with open('../../taxonomy/gpt4o_edited.json', 'r') as file:
    gpt4o_edited_taxonomy = json.load(file)    

In [ ]:
print(format_taxonomy(gpt4o_edited_taxonomy))

In [ ]:
# Test the function with the openai_taxonomy data

shap_features = {
    'feature': df_shap["feature_name"].to_list()
}
features_to_find = shap_features["feature"]

taxonomy_model = [gpt4o_taxonomy, llama3_taxonomy, gemma2_taxonomy, deepseek_taxonomy, gpt4o_edited_taxonomy, ]
index = 4

match index:
    case 0:
        print("GPT-4o")
    case 1:
        print("Llama 3.3")
    case 2:
        print("Gemma2")
    case 3:
        print("DeepSeek")
    case 4:
        print("GPT-4o Edited")
    case _:
        print("Unknown model")

try:
    # Get paths for all selected features
    feature_paths = get_taxonomy_paths(taxonomy_model[index], features_to_find)

    # Print the results in a formatted way, removing "Sensor_Reading --> " from the start
    print("\n=== Taxonomy Paths ===\n")
    for feature, path in feature_paths.items():
        if path.startswith("Sensor_Readings --> "):
            path = path[len("Sensor_Readings --> "):]
        print(f"{path}")
except IndexError as e:
    print("Taxonomy model index is out of reach")

In [ ]:
df_shap

<br> <br> <br>

#### Map the top n SHAP to causal graph - Simple visualization

In [ ]:
def visualize_and_json_shap_causal(df_shap, structured_adjacency):
    """
    Visualize the subgraph of the causal graph containing top SHAP features and return the subgraph as JSON.

    Args:
        df_shap (pd.DataFrame): DataFrame with a 'feature_name' column.
        structured_adjacency (list): List of dicts with 'source_feature' and 'target_feature' keys.

    Returns:
        str: JSON string of the subgraph.
    """
    import matplotlib.pyplot as plt
    import networkx as nx
    import json

    top_features = set(df_shap["feature_name"])
    sub_edges = []
    for edge in structured_adjacency:
        if (edge["source_feature"] in top_features) or (edge["target_feature"] in top_features):
            sub_edges.append((edge["source_feature"], edge["target_feature"]))

    G = nx.DiGraph()
    G.add_edges_from(sub_edges)

    node_colors = ['orange' if node in top_features else 'skyblue' for node in G.nodes()]
    edge_colors = ['red' if (u in top_features and v in top_features) else 'gray' for u, v in G.edges()]

    # Use graphviz_layout if available for better readability, else fallback to spring_layout
    try:
        pos = nx.nx_agraph.graphviz_layout(G, prog='dot')
    except Exception:
        pos = nx.spring_layout(G, seed=42, k=0.8, iterations=200)

    plt.figure(figsize=(max(12, len(G.nodes()) * 0.7), max(8, len(G.nodes()) * 0.5)))
    nx.draw(
        G, pos,
        with_labels=True,
        node_color=node_colors,
        edge_color=edge_colors,
        node_size=1400,
        font_size=11,
        arrowsize=22,
        linewidths=2,
        font_weight='bold'
    )
    plt.title("Subgraph: Top SHAP Features in Causal Graph", fontsize=16)
    plt.tight_layout()
    plt.show()

    subgraph_json = {
        "nodes": [
            {"id": node, "highlight": node in top_features}
            for node in G.nodes()
        ],
        "edges": [
            {
                "source": u,
                "target": v,
                "highlight": (u in top_features and v in top_features)
            }
            for u, v in G.edges()
        ]
    }
    return json.dumps(subgraph_json, indent=2)



In [ ]:
# Example usage:
result = visualize_and_json_shap_causal(df_shap, structured_adjacency)

<br> <br> <br>

## Visualize The subgraph without Target variable- Advanced Visualization

Mapping the features from SHAP to the Causal graph that is built base don the whole dataset

In [ ]:
def create_interactive_visualization(df_shap, structured_adjacency, top_n_sub_nodes=None, filename="interactive_causal_graph.html", width="100%", height="1000px"):
    """
    Creates an interactive HTML visualization of the causal graph containing top SHAP features.
    
    This version includes the top_n_sub_nodes filtering logic and creates an interactive visualization
    using pyvis instead of matplotlib.

    Args:
        df_shap (pd.DataFrame): DataFrame with a 'feature_name' column identifying the core features.
        structured_adjacency (list): List of dicts with 'source_feature', 'target_feature', 
                                     and 'effect_strength' keys.
        top_n_sub_nodes (int, optional): The number of top related nodes to include in the graph,
                                         ranked by absolute effect_strength. If None, all directly
                                         connected nodes are included. Defaults to None.
        filename (str): The name of the output HTML file. Defaults to "interactive_causal_graph.html".
        width (str): Width of the visualization. Defaults to "100%".
        height (str): Height of the visualization. Defaults to "1000px".
    
    Returns:
        str: JSON string of the subgraph (same as original function).
    """
    import networkx as nx
    import json
    import pandas as pd
    from pyvis.network import Network

    top_features = set(df_shap["feature_name"])
    
    # Store original effect strengths for JSON output
    edge_strengths = {}
    
    # If top_n_sub_nodes is specified, filter to the most influential neighbors
    if top_n_sub_nodes is not None and isinstance(top_n_sub_nodes, int) and top_n_sub_nodes >= 0:
        # 1. Identify all sub-nodes connected to top_features and rank them by effect strength
        sub_node_strengths = {}
        for edge in structured_adjacency:
            source, target = edge["source_feature"], edge["target_feature"]
            strength = abs(edge.get("effect_strength", 0))  # Use absolute strength for ranking

            # Find sub-nodes (nodes not in top_features) connected to top_features
            if source in top_features and target not in top_features:
                sub_node = target
                sub_node_strengths[sub_node] = max(sub_node_strengths.get(sub_node, 0), strength)
            elif target in top_features and source not in top_features:
                sub_node = source
                sub_node_strengths[sub_node] = max(sub_node_strengths.get(sub_node, 0), strength)
        
        # 2. Select the top N sub-nodes
        sorted_sub_nodes = sorted(sub_node_strengths.items(), key=lambda item: item[1], reverse=True)
        top_sub_nodes_set = {node for node, strength in sorted_sub_nodes[:top_n_sub_nodes]}
        
        # 3. The final set of nodes to display includes top features and the selected top sub-nodes
        allowed_nodes = top_features.union(top_sub_nodes_set)
        
        # 4. Filter edges where both source and target are in our allowed set
        sub_edges = []
        edge_labels = {}
        for edge in structured_adjacency:
            source, target = edge["source_feature"], edge["target_feature"]
            if source in allowed_nodes and target in allowed_nodes:
                sub_edges.append((source, target))
                effect_strength = edge.get("effect_strength", 0)
                edge_labels[(source, target)] = f'{effect_strength:.3f}'
                edge_strengths[(source, target)] = effect_strength  # Store original value
    else:
        # Original logic: include all nodes directly connected to any top_feature
        sub_edges = []
        edge_labels = {}
        for edge in structured_adjacency:
            if (edge["source_feature"] in top_features) or (edge["target_feature"] in top_features):
                source, target = edge["source_feature"], edge["target_feature"]
                sub_edges.append((source, target))
                effect_strength = edge.get("effect_strength", 0)
                edge_labels[(source, target)] = f'{effect_strength:.3f}'
                edge_strengths[(source, target)] = effect_strength  # Store original value

    # Create NetworkX graph
    G = nx.DiGraph()
    G.add_edges_from(sub_edges)
    
    # Add edge labels as attributes to the graph
    for (u, v), label in edge_labels.items():
        if G.has_edge(u, v):
            G.edges[u, v]['label'] = label

    # If the graph is empty, return an empty JSON and skip visualization
    if not G.nodes():
        print("Warning: The resulting graph is empty. No relationships found based on the criteria.")
        return json.dumps({"nodes": [], "edges": []}, indent=2)

    # Create interactive visualization using pyvis
    print(f"Creating interactive visualization with {len(G.nodes())} nodes and {len(G.edges())} edges...")
    
    net = Network(height=height, width=width, notebook=False, directed=True, cdn_resources='in_line')

    # Set physics options for better layout and interactivity
    net.set_options("""
    var options = {
      "physics": {
        "enabled": true,
        "repulsion": { 
          "centralGravity": 0.2, 
          "springLength": 200, 
          "nodeDistance": 300,
          "damping": 0.1
        },
        "minVelocity": 0.75,
        "solver": "repulsion",
        "timestep": 0.22,
        "stabilization": {"iterations": 150}
      },
      "edges": {
        "arrows": {
          "to": {"enabled": true, "scaleFactor": 1.5}
        },
        "smooth": {
          "enabled": true,
          "type": "continuous"
        }
      },
      "interaction": {
        "dragNodes": true,
        "dragView": true,
        "zoomView": true,
        "selectConnectedEdges": true,
        "hover": true,
        "multiselect": true,
        "keyboard": {
          "enabled": true
        }
      },
      "manipulation": {
        "enabled": false
      }
    }
    """)

    # Add nodes to the pyvis network with enhanced interactivity
    for node in G.nodes():
        is_top = node in top_features
        net.add_node(
            node, 
            label=node, 
            color='orange' if is_top else 'skyblue',
            shape='box', 
            shadow=True, 
            font={'size': 18, 'color': 'black', 'strokeWidth': 2, 'strokeColor': 'white'},
            size=25,
            borderWidth=2,
            borderWidthSelected=4,
            title=f"{'🎯 Top SHAP Feature' if is_top else '🔗 Related Feature'}: {node}<br/>Click and drag to move!"  # Enhanced tooltip
        )

    # Add edges to the pyvis network with enhanced styling
    for u, v in G.edges():
        label = G.edges[u, v].get('label', '')
        is_highlight = (u in top_features and v in top_features)
        net.add_edge(
            u, v, 
            label=label, 
            color='red' if is_highlight else 'gray',
            width=4 if is_highlight else 2,
            title=f"📊 Effect strength: {label}<br/>From: {u} → To: {v}",  # Enhanced tooltip
            font={'size': 12, 'color': 'blue', 'strokeWidth': 1, 'strokeColor': 'white'}
        )

    # Generate and save HTML content with proper encoding
    html_content = net.generate_html()
    try:
        with open(filename, "w", encoding="utf-8") as f:
            f.write(html_content)
        print(f"[SUCCESS] Interactive graph saved to '{filename}'. Open this file in your browser.")
    except Exception as e:
        print(f"[ERROR] Could not save the HTML file. Reason: {e}")
        return json.dumps({"nodes": [], "edges": []}, indent=2)

    # Generate JSON output (same as original function)
    subgraph_json = {
        "nodes": [
            {"id": node, "highlight": node in top_features}
            for node in G.nodes()
        ],
        "edges": [
            {
                "source": u,
                "target": v,
                "highlight": (u in top_features and v in top_features),
                "effect_strength": edge_strengths.get((u, v), 0.0)  # Use original numeric value
            }
            for u, v in G.edges()
        ]
    }
    
    return json.dumps(subgraph_json, indent=2)


In [ ]:
# Default larger size (1000px height)
# result = create_interactive_visualization(df_shap, structured_adjacency, top_n_sub_nodes=7)

# Custom large size
# result = create_interactive_visualization(df_shap, structured_adjacency, top_n_sub_nodes=7, width="1400px", height="1200px")

# Full screen
result = create_interactive_visualization(df_shap, structured_adjacency,filename= 'interactive_causal_graph_without_target.html', top_n_sub_nodes=None, width="100vw", height="100vh")


<br> <br> <br>

## Visualize The subgraph - Advanced Visualization - Target included


In [ ]:
def create_interactive_visualization_with_target(df_shap, structured_adjacency, top_n_sub_nodes=None, filename="interactive_causal_graph.html", width="100%", height="1000px"):
    """
    Creates an interactive HTML visualization of the causal graph containing top SHAP features.
    
    This version includes the top_n_sub_nodes filtering logic and creates an interactive visualization
    using pyvis instead of matplotlib.

    Args:
        df_shap (pd.DataFrame): DataFrame with a 'feature_name' column identifying the core features.
        structured_adjacency (list): List of dicts with 'source_feature', 'target_feature', 
                                     and 'effect_strength' keys.
        top_n_sub_nodes (int, optional): The number of top related nodes to include in the graph,
                                         ranked by absolute effect_strength. If None, all directly
                                         connected nodes are included. Defaults to None.
        filename (str): The name of the output HTML file. Defaults to "interactive_causal_graph.html".
        width (str): Width of the visualization. Defaults to "100%".
        height (str): Height of the visualization. Defaults to "1000px".
    
    Returns:
        str: JSON string of the subgraph (same as original function).
    """
    import networkx as nx
    import json
    import pandas as pd
    from pyvis.network import Network

    top_features = set(df_shap["feature_name"])
    target_variable = 'target'  # Define the target variable name
    
    # Store original effect strengths for JSON output
    edge_strengths = {}
    
    # If top_n_sub_nodes is specified, filter to the most influential relationships
    if top_n_sub_nodes is not None and isinstance(top_n_sub_nodes, int) and top_n_sub_nodes >= 0:
        # 1. Find all edges connecting top_features to other nodes (sub-nodes), excluding target
        relevant_edges = []
        for edge in structured_adjacency:
            source, target = edge["source_feature"], edge["target_feature"]
            strength = abs(edge.get("effect_strength", 0))  # Use absolute strength for ranking
            
            # Find relationships where one node is in top_features and the other is not (excluding target variable)
            if source in top_features and target not in top_features and target != target_variable:
                relevant_edges.append({
                    'top_feature': source,
                    'sub_node': target,
                    'strength': strength,
                    'original_edge': edge
                })
            elif target in top_features and source not in top_features and source != target_variable:
                relevant_edges.append({
                    'top_feature': target,
                    'sub_node': source,
                    'strength': strength,
                    'original_edge': edge
                })
        
        # 2. Sort all relationships by effect strength and select top N
        sorted_relationships = sorted(relevant_edges, key=lambda x: x['strength'], reverse=True)
        top_relationships = sorted_relationships[:top_n_sub_nodes]
        
        # 3. Get the set of sub-nodes from top N relationships
        top_sub_nodes_set = {rel['sub_node'] for rel in top_relationships}
        
        print(f"Selected top {len(top_relationships)} relationships with sub-nodes: {top_sub_nodes_set}")
        for i, rel in enumerate(top_relationships, 1):
            print(f"  {i}. {rel['top_feature']} ↔ {rel['sub_node']} (strength: {rel['strength']:.4f})")
        
        # 4. The final set of nodes includes top features, selected sub-nodes, AND target variable
        allowed_nodes = top_features.union(top_sub_nodes_set).union({target_variable})
        
        # 5. Filter edges where both source and target are in our allowed set
        sub_edges = []
        edge_labels = {}
        for edge in structured_adjacency:
            source, target = edge["source_feature"], edge["target_feature"]
            if source in allowed_nodes and target in allowed_nodes:
                sub_edges.append((source, target))
                effect_strength = edge.get("effect_strength", 0)
                edge_labels[(source, target)] = f'{effect_strength:.3f}'
                edge_strengths[(source, target)] = effect_strength  # Store original value
    else:
        # Original logic: include all nodes directly connected to any top_feature, PLUS target variable
        sub_edges = []
        edge_labels = {}
        allowed_nodes = set()
        
        for edge in structured_adjacency:
            source, target = edge["source_feature"], edge["target_feature"]
            # Include edges connected to top_features OR target variable
            if (source in top_features) or (target in top_features) or (source == target_variable) or (target == target_variable):
                sub_edges.append((source, target))
                effect_strength = edge.get("effect_strength", 0)
                edge_labels[(source, target)] = f'{effect_strength:.3f}'
                edge_strengths[(source, target)] = effect_strength  # Store original value
                allowed_nodes.add(source)
                allowed_nodes.add(target)

    # Create NetworkX graph
    G = nx.DiGraph()
    G.add_edges_from(sub_edges)
    
    # Add edge labels as attributes to the graph
    for (u, v), label in edge_labels.items():
        if G.has_edge(u, v):
            G.edges[u, v]['label'] = label

    # If the graph is empty, return an empty JSON and skip visualization
    if not G.nodes():
        print("Warning: The resulting graph is empty. No relationships found based on the criteria.")
        return json.dumps({"nodes": [], "edges": []}, indent=2)

    # Create interactive visualization using pyvis
    print(f"Creating interactive visualization with {len(G.nodes())} nodes and {len(G.edges())} edges...")
    
    net = Network(height=height, width=width, notebook=False, directed=True, cdn_resources='in_line')

    # Set physics options for better layout and interactivity
    net.set_options("""
    var options = {
      "physics": {
        "enabled": true,
        "repulsion": { 
          "centralGravity": 0.2, 
          "springLength": 200, 
          "nodeDistance": 300,
          "damping": 0.1
        },
        "minVelocity": 0.75,
        "solver": "repulsion",
        "timestep": 0.22,
        "stabilization": {"iterations": 150}
      },
      "edges": {
        "arrows": {
          "to": {"enabled": true, "scaleFactor": 1.5}
        },
        "smooth": {
          "enabled": true,
          "type": "continuous"
        }
      },
      "interaction": {
        "dragNodes": true,
        "dragView": true,
        "zoomView": true,
        "selectConnectedEdges": true,
        "hover": true,
        "multiselect": true,
        "keyboard": {
          "enabled": true
        }
      },
      "manipulation": {
        "enabled": false
      }
    }
    """)

    # Add nodes to the pyvis network with enhanced interactivity
    for node in G.nodes():
        is_top = node in top_features
        is_target = node == target_variable  
        
        # Determine node styling based on type
        if is_target:
            color = 'red'
            node_type = '🎯 TARGET VARIABLE'
            size = 35  # Larger size for target
        elif is_top:
            color = 'orange'
            node_type = '⭐ Top SHAP Feature'
            size = 25
        else:
            color = 'skyblue'
            node_type = '🔗 Related Feature'
            size = 20
            
        net.add_node(
            node, 
            label=node, 
            color=color,
            shape='box', 
            shadow=True, 
            font={'size': 18, 'color': 'black', 'strokeWidth': 2, 'strokeColor': 'white'},
            size=size,
            borderWidth=3 if is_target else 2,
            borderWidthSelected=5 if is_target else 4,
            title=f"{node_type}: {node}<br/>Click and drag to move!"  # Enhanced tooltip
        )

    # Add edges to the pyvis network with enhanced styling
    for u, v in G.edges():
        label = G.edges[u, v].get('label', '')
        is_highlight = (u in top_features and v in top_features)
        is_target_edge = (u == target_variable or v == target_variable)
        
        # Special styling for edges connected to target
        if is_target_edge:
            edge_color = 'darkred'
            edge_width = 5
            edge_title = f"🎯 TARGET CONNECTION - Effect strength: {label}<br/>From: {u} → To: {v}"
        elif is_highlight:
            edge_color = 'red'
            edge_width = 4
            edge_title = f"📊 SHAP-SHAP Connection - Effect strength: {label}<br/>From: {u} → To: {v}"
        else:
            edge_color = 'gray'
            edge_width = 2
            edge_title = f"📊 Effect strength: {label}<br/>From: {u} → To: {v}"
            
        net.add_edge(
            u, v, 
            label=label, 
            color=edge_color,
            width=edge_width,
            title=edge_title,  # Enhanced tooltip
            font={'size': 12, 'color': 'blue', 'strokeWidth': 1, 'strokeColor': 'white'}
        )

    # Generate and save HTML content with proper encoding
    html_content = net.generate_html()
    try:
        with open(filename, "w", encoding="utf-8") as f:
            f.write(html_content)
        print(f"[SUCCESS] Interactive graph saved to '{filename}'. Open this file in your browser.")
    except Exception as e:
        print(f"[ERROR] Could not save the HTML file. Reason: {e}")
        return json.dumps({"nodes": [], "edges": []}, indent=2)

    # Generate JSON output (same as original function, but include target info)
    subgraph_json = {
        "nodes": [
            {
                "id": node, 
                "highlight": node in top_features,
                "is_target": node == target_variable,
                "node_type": "target" if node == target_variable else ("shap_feature" if node in top_features else "related_feature")
            }
            for node in G.nodes()
        ],
        "edges": [
            {
                "source": u,
                "target": v,
                "highlight": (u in top_features and v in top_features),
                "is_target_connection": (u == target_variable or v == target_variable),
                "effect_strength": edge_strengths.get((u, v), 0.0)  # Use original numeric value
            }
            for u, v in G.edges()
        ]
    }
    
    return json.dumps(subgraph_json, indent=2)

In [ ]:
# Example usage with custom size:
# For a larger plot:
# result = create_interactive_visualization_with_target(df_shap, structured_adjacency, top_n_sub_nodes=7, filename="large_graph.html", width="1400px", height="1200px")
# 
# For full screen:
result = create_interactive_visualization_with_target(df_shap, structured_adjacency, top_n_sub_nodes=None, filename="interactive_causal_graph_with_target.html", width="100vw", height="100vh")

In [ ]:
# print(result)

<br> <br> <br>

---

<br> <br> <br>

## Counterfactual

In [ ]:
def run_advanced_counterfactuals(
    df_shap,
    structured_adjacency,
    base_row,
    model_dict,
    num_runs=10,
    perturbation_scale=0.1,
    perturbation_method='gaussian',  # 'gaussian', 'uniform', or 'fixed'
    fixed_value=None,                # Used if method is 'fixed'
    random_seed=None,
    verbose=True
):
    """
    Advanced counterfactual generator for SHAP-causal subgraph.
    Tracks feature changes, supports multiple perturbation methods, and logs model predictions.

    Args:
        df_shap (pd.DataFrame): Top SHAP features DataFrame.
        structured_adjacency (list): Causal graph adjacency list.
        base_row (pd.Series): The base row to perturb.
        model_dict (dict): Model dictionary with 'model' and 'feature_names'.
        num_runs (int): Number of counterfactual runs.
        perturbation_scale (float): Scale of random perturbation for features.
        perturbation_method (str): 'gaussian', 'uniform', or 'fixed'.
        fixed_value (float): Value to set if method is 'fixed'.
        random_seed (int or None): Random seed for reproducibility.
        verbose (bool): If True, print detailed logs.

    Returns:
        consistent_nodes (dict): Nodes with consistent values and their value.
        cf_df (pd.DataFrame): All counterfactual results for inspection.
        logs (list): List of dicts with feature changes and predictions for each run.
    """
    import numpy as np
    import pandas as pd

    if random_seed is not None:
        np.random.seed(random_seed)

    # Get the subgraph nodes (features) from the SHAP-causal mapping
    top_features = set(df_shap['feature_name'])
    subgraph_nodes = set()
    for edge in structured_adjacency:
        if (edge['source_feature'] in top_features) or (edge['target_feature'] in top_features):
            subgraph_nodes.add(edge['source_feature'])
            subgraph_nodes.add(edge['target_feature'])
    subgraph_nodes = [n for n in subgraph_nodes if n in base_row.index]

    base_values = base_row[subgraph_nodes].copy()
    model = model_dict['model']
    feature_names = model_dict['feature_names']

    counterfactual_results = []
    logs = []

    for i in range(num_runs):
        perturbed = base_values.copy()
        feature_changes = []
        if verbose:
            print(f"\nCounterfactual run {i+1}:")
        for feat in top_features:
            if feat in perturbed.index:
                old_value = perturbed[feat]
                # Choose perturbation method
                if perturbation_method == 'gaussian':
                    std = np.std(df_shap[df_shap['feature_name'] == feat]['feature_importance'])
                    std = std if std > 0 else 0.05
                    noise = np.random.normal(0, perturbation_scale * std)
                    new_value = old_value + noise
                elif perturbation_method == 'uniform':
                    noise = np.random.uniform(-perturbation_scale, perturbation_scale)
                    new_value = old_value + noise
                elif perturbation_method == 'fixed':
                    new_value = fixed_value if fixed_value is not None else old_value
                    noise = new_value - old_value
                else:
                    raise ValueError("Unknown perturbation_method")
                perturbed[feat] = new_value
                feature_changes.append({
                    'feature': feat,
                    'old_value': old_value,
                    'new_value': new_value,
                    'perturbation': noise
                })
                if verbose:
                    print(f"  Changed feature '{feat}': {old_value:.6f} -> {new_value:.6f} (Δ={noise:.6f})")
        # Prepare input for model prediction (fill missing features with base_row if needed)
        full_input = base_row.copy()
        for feat in perturbed.index:
            full_input[feat] = perturbed[feat]
        X_input = full_input[feature_names].to_frame().T
        X_input = X_input.apply(pd.to_numeric, errors='coerce')
        try:
            y_pred = model.predict(X_input)[0]
        except Exception:
            y_pred = None
        counterfactual_results.append(perturbed)
        logs.append({
            'run': i+1,
            'feature_changes': feature_changes,
            'model_prediction': y_pred
        })

    cf_df = pd.DataFrame(counterfactual_results)

    # Find consistent nodes: nodes whose value is (almost) the same across all runs
    consistent_nodes = {}
    for col in cf_df.columns:
        values = cf_df[col].values
        if np.allclose(values, values[0], atol=1e-4):
            consistent_nodes[col] = values[0]

    if verbose:
        print(f"\nRan {num_runs} counterfactuals. Consistent nodes in the sub-causal graph:")
        for node, value in consistent_nodes.items():
            print(f"  {node}: {value}")

    # Summary statistics
    if verbose:
        print("\n=== Counterfactual Summary ===")
        for feat in top_features:
            changes = [chg for log in logs for chg in log['feature_changes'] if chg['feature'] == feat]
            if changes:
                avg_perturb = np.mean([abs(chg['perturbation']) for chg in changes])
                print(f"Feature '{feat}': changed {len(changes)} times, avg |perturbation|={avg_perturb:.4f}")

        preds = [log['model_prediction'] for log in logs if log['model_prediction'] is not None]
        if preds:
            unique, counts = np.unique(preds, return_counts=True)
            print("Model predictions across runs:", dict(zip(unique, counts)))

    return consistent_nodes, cf_df, logs



In [ ]:
# Example usage:
consistent_nodes, all_counterfactuals, logs = run_advanced_counterfactuals(
    df_shap=df_shap,
    structured_adjacency=structured_adjacency,
    base_row=random_row,
    model_dict=model,
    num_runs=100,
    perturbation_scale=0.1,
    perturbation_method='gaussian',
    random_seed=None,
    verbose=True
)

<br> <br> <br> <br>


The output  
**Model predictions across runs: {0: 27, 3: 73}**  
means that, after generating 100 counterfactual samples:

- The model predicted class **0** for **27** of the counterfactuals.
- The model predicted class **3** for **73** of the counterfactuals.

**Interpretation:**  
This shows how the model’s output changes when you perturb the most important features. In this case, most counterfactuals led to class 3, but a significant number led to class 0. This suggests that the model’s prediction for this instance is sensitive to the perturbed features and can switch between these two classes depending on the feature values.

In [ ]:
def generate_counterfactual_summary(consistent_nodes, all_counterfactuals, logs, max_features=3):
    """
    Create a concise, actionable summary of counterfactual analysis
    by combining consistent_nodes, all_counterfactuals, and logs.

    Args:
        consistent_nodes (dict): Features stable across all counterfactuals.
        all_counterfactuals (pd.DataFrame): DataFrame of all counterfactual samples.
        logs (list): List of dicts with feature changes and predictions per run.
        max_features (int): Max number of top features to highlight in summary.

    Returns:
        str: Human-readable summary for LLM prompt.
    """
    # 1. Stable features
    stable = list(consistent_nodes.keys())
    if stable:
        stable_str = f"Stable features (not needing change): {', '.join(map(str, stable[:max_features]))}."
    else:
        stable_str = "No features remained completely stable across counterfactuals."

    # 2. Most frequently changed features
    feature_change_counts = {}
    for log in logs:
        for chg in log['feature_changes']:
            f = chg['feature']
            # Convert numpy arrays to string for hashing
            if isinstance(f, (np.ndarray, list)):
                f = str(f)
            feature_change_counts[f] = feature_change_counts.get(f, 0) + 1
    if feature_change_counts:
        sorted_features = sorted(feature_change_counts.items(), key=lambda x: x[1], reverse=True)
        top_changed = [f"{f} (changed {c}x)" for f, c in sorted_features[:max_features]]
        changed_str = f"Most sensitive features: {', '.join(top_changed)}."
    else:
        changed_str = "No feature changes detected in counterfactuals."

    # 3. Model prediction diversity
    preds = []
    for log in logs:
        pred = log.get('model_prediction')
        # Convert numpy arrays to scalar if needed
        if isinstance(pred, np.ndarray):
            if pred.size == 1:
                pred = pred.item()
            else:
                pred = str(pred)
        if pred is not None:
            preds.append(pred)
    if preds:
        from collections import Counter
        pred_counts = Counter(preds)
        pred_str = "Model predictions across counterfactuals: " + ", ".join(
            [f"{k}: {v}" for k, v in pred_counts.items()]
        ) + "."
    else:
        pred_str = "No model predictions available for counterfactuals."

    # 4. Minimal actionable insight (optional: what changes flip prediction)
    flip_insight = ""
    if preds and len(set(preds)) > 1:
        flip_insight = "Some feature changes can flip the model prediction, indicating sensitivity to these features."
    elif preds:
        flip_insight = "Model prediction is robust to most feature changes."
    else:
        flip_insight = ""

    # Combine all
    summary = f"{stable_str} {changed_str} {pred_str} {flip_insight}".strip()
    return summary

In [ ]:
counterfactual_summary = generate_counterfactual_summary(consistent_nodes=consistent_nodes, all_counterfactuals=all_counterfactuals, logs= logs, max_features=len(df_shap["feature_name"]))

In [ ]:
display(Markdown(counterfactual_summary))

In [ ]:
feature_mappings = {
    # Raw sensor readings
    'volt': 'Current voltage reading',
    'rotate': 'Current rotation speed',
    'pressure': 'Current pressure level',
    'vibration': 'Current vibration intensity',
    'errorID': 'Current error code',
    'comp': 'Current compressor state',
    
    # Voltage features
    'volt_lag_1h': 'Voltage level 1 hour ago',
    'volt_lag_6h': 'Voltage level 6 hours ago',
    'volt_lag_12h': 'Voltage level 12 hours ago',
    'volt_lag_24h': 'Voltage level 24 hours ago',
    'volt_mean_24h': 'Average voltage over last 24 hours',
    'volt_std_24h': 'Voltage variability over last 24 hours',
    'volt_min_24h': 'Minimum voltage in last 24 hours',
    'volt_max_24h': 'Maximum voltage in last 24 hours',
    'volt_mean_6h': 'Average voltage over last 6 hours',
    'volt_std_6h': 'Voltage variability over last 6 hours',
    
    # Rotation features
    'rotate_lag_1h': 'Rotation speed 1 hour ago',
    'rotate_lag_6h': 'Rotation speed 6 hours ago',
    'rotate_lag_12h': 'Rotation speed 12 hours ago',
    'rotate_lag_24h': 'Rotation speed 24 hours ago',
    'rotate_mean_24h': 'Average rotation speed over last 24 hours',
    'rotate_std_24h': 'Rotation speed variability over last 24 hours',
    'rotate_min_24h': 'Minimum rotation speed in last 24 hours',
    'rotate_max_24h': 'Maximum rotation speed in last 24 hours',
    'rotate_mean_6h': 'Average rotation speed over last 6 hours',
    'rotate_std_6h': 'Rotation speed variability over last 6 hours',
    
    # Pressure features
    'pressure_lag_1h': 'Pressure level 1 hour ago',
    'pressure_lag_6h': 'Pressure level 6 hours ago',
    'pressure_lag_12h': 'Pressure level 12 hours ago',
    'pressure_lag_24h': 'Pressure level 24 hours ago',
    'pressure_mean_24h': 'Average pressure over last 24 hours',
    'pressure_std_24h': 'Pressure variability over last 24 hours',
    'pressure_min_24h': 'Minimum pressure in last 24 hours',
    'pressure_max_24h': 'Maximum pressure in last 24 hours',
    'pressure_mean_6h': 'Average pressure over last 6 hours',
    'pressure_std_6h': 'Pressure variability over last 6 hours',
    
    # Vibration features
    'vibration_lag_1h': 'Vibration intensity 1 hour ago',
    'vibration_lag_6h': 'Vibration intensity 6 hours ago',
    'vibration_lag_12h': 'Vibration intensity 12 hours ago',
    'vibration_lag_24h': 'Vibration intensity 24 hours ago',
    'vibration_mean_24h': 'Average vibration intensity over last 24 hours',
    'vibration_std_24h': 'Vibration variability over last 24 hours',
    'vibration_min_24h': 'Minimum vibration in last 24 hours',
    'vibration_max_24h': 'Maximum vibration in last 24 hours',
    'vibration_mean_6h': 'Average vibration intensity over last 6 hours',
    'vibration_std_6h': 'Vibration variability over last 6 hours',
    
    # Error features
    'errorID_lag_1h': 'Error code 1 hour ago',
    'errorID_lag_6h': 'Error code 6 hours ago',
    'errorID_lag_12h': 'Error code 12 hours ago',
    
    # Compressor state features
    'comp_lag_1h': 'Compressor state 1 hour ago',
    'comp_lag_6h': 'Compressor state 6 hours ago',
    'comp_lag_12h': 'Compressor state 12 hours ago',
    
    # Event counts
    'error_count_6h': 'Number of errors in last 6 hours',
    'error_count_24h': 'Number of errors in last 24 hours',
    'maint_count_6h': 'Number of maintenance events in last 6 hours',
    'maint_count_24h': 'Number of maintenance events in last 24 hours',
    
    # Time features
    'hour': 'Hour of day (0-23)',
    'day_of_week': 'Day of week (0=Monday, 6=Sunday)',
    'is_weekend': 'Weekend indicator (1=weekend, 0=weekday)',
    'is_working_hours': 'Working hours indicator (1=working hours, 0=non-working)',
    
    # Time since events
    'hours_since_maint': 'Hours since last maintenance',
    'hours_since_error': 'Hours since last error'
}


failure_mapping = {
    1: "component 1",
    2: "component 2",
    3: "component 3",
    4: "component 4"
}

In [ ]:
# random_row_full.to_frame().T[["target"]]

# Extract the target value and map it to the failure description

failure_mapping.get(random_row_full.to_frame().T["target"].values[0], "Unknown failure")






In [ ]:
def generate_explanation_actionable_recommendation_openai(persona, failure_type, feature_values, shap_top_features, features_hierarchy, 
                                                          causal_graph, counterfactual_summary, openai_api_key, response_folder, 
                                                          random_row_index_id, feature_map):
    """
    Build a prompt for LLMs to generate an explanation and actionable recommendation
    using SHAP, causal graph, and counterfactual analysis.

    Args:
        persona (str): The target persona (e.g., "Operator", "Data Scientist").
        failure_type (str): The type of failure detected.
        failure_description (str): Description of the failure.
        feature_values (dict or str): Sensor/feature values for the instance.
        shap_top_features (list or str): Top SHAP features for the instance.
        causal_graph (dict or str): Causal relationships (JSON or string).
        counterfactual_summary (str): Summary of counterfactual analysis.
        recommended_action (str): Recommended action(s) for the failure.
        important_actions (str): Required actions that must be included.

    Returns:
        str: The formatted prompt.
    """
    prompt = f"""
                **Task:**

                You are assisting a **{persona}** in understanding why a predictive maintenance system will detect a failure {failure_type} and what actions should be taken to prevent it from happening.

                You have access to:
                - Sensor values (features): {feature_values}                
                - SHAP feature importances: {shap_top_features}
                - Features Hierarchy: {features_hierarchy}
                - Causal relationships (with effect_strengths) in JSON: {causal_graph}
                - Counterfactual analysis results: {counterfactual_summary}

                ---

                **Instructions:**

                Adapt your explanation and advice for a **{persona}**:
                - **Machine Operator:** Use plain language, avoid technical jargon, and focus on clear, actionable steps.
                - **Data Scientist:** Emphasize data patterns, causal reasoning, and model-inferred insights.
                
                
                Provide your response in two clearly labeled parts:

                **Important:** Consider the features hierarchy for generating explnation and actionable recommendation. 
                    Example: 'vibration --> vibration_std_24h' 
                    In this example "vibration_std_24h" is the child of "vibration"
                    

                ---

                ### a. Explanation of the Failure (max 100 words)

                - Summarize **what led to the failure** for a **{persona}**.
                - Reference the most important features (from SHAP: {shap_top_features}) and translate them into real-world terms if needed.
                - Use the **causal graph** ({causal_graph}) to highlight which factors had the strongest causal impact (by effect_strength).
                - Mention what the counterfactual analysis ({counterfactual_summary}) reveals about which feature changes could have prevented the failure.
                - Adjust technicality to suit the persona.
                - Use the {feature_map} to map the feature names with their human understandable pair when for both expalnation and actionable recommendation.


                ---

                ### b. Actionable Recommendation (max 100 words)

                - Use insights from counterfactuals {counterfactual_summary} to suggest the **minimal or most effective changes** that would reduce failure risk.
                - Make the guidance clear, specific, and relevant to the persona's role.

                ---

                **Important Notes:**

                - Your explanation and recommendation **must reflect all input parameters**: `{feature_values}`, `{shap_top_features}`, `{features_hierarchy}`, `{causal_graph}` and `{counterfactual_summary}`
                - Be clear, helpful, and context-aware. The message should feel tailored for the {persona}.
                """

    openai_url = "https://api.openai.com/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {openai_api_key}",
        "Content-Type": "application/json",
    }
    data = {
        "model": "gpt-4o",  # Replace with your model choice if needed
        "messages": [{"role": "user", "content": prompt}],
        # "max_tokens": 150
    }

    response = requests.post(openai_url, headers=headers, json=data)

    current_date = datetime.now().strftime("%d%m%Y")
    eval_folder = (
        os.path.join(response_folder, current_date)
        if current_date in response_folder
        else response_folder
    )

    # Save the response to the specified path
    response_file_path = os.path.join(
        # response_folder, f"explanation_{random_row_index_id}.md"
        eval_folder,
        f"explanation_{random_row_index_id}.md",
    )
    explanation_text = response.json()["choices"][0]["message"]["content"]
    with open(response_file_path, "w", encoding="utf-8") as md_file:
        md_file.write(explanation_text)

    return response.json()


In [ ]:
# persona, failure_type, feature_values, shap_top_features, features_hierarchy, causal_graph, counterfactual_summary, openai_api_key
# persona = "machine operator" 
persona = "data scientist" 

if persona == "machine operator":
    response_folder_path = f"../../llm_response/azure_pm/machine_{machine_number}/machine_operator/"
    if not os.path.exists(response_folder_path):
        os.makedirs(response_folder_path, exist_ok=True)
elif persona == "data scientist":
    response_folder_path = f"../../llm_response/azure_pm/machine_{machine_number}/data_scientist/"
    if not os.path.exists(response_folder_path):
        os.makedirs(response_folder_path, exist_ok=True)
    

response = generate_explanation_actionable_recommendation_openai(
            persona = persona,  # Pass the persona parameter
            failure_type = failure_mapping.get(random_row_full.to_frame().T["target"].values[0], "Unknown failure"),
            feature_values = random_row.to_dict(),         
            shap_top_features = df_shap["feature_name"], 
            features_hierarchy=list(feature_paths.values()),
            causal_graph=structured_adjacency,
            counterfactual_summary= counterfactual_summary,
            openai_api_key=openai_api_key,
            response_folder= response_folder_path,
            random_row_index_id=random_row.to_frame().T.index.values[0],
            feature_map=feature_mappings
        )

In [ ]:
explanation_file = os.path.join(response_folder_path, f"explanation_{random_row.to_frame().T.index.values[0]}.md")
with open(explanation_file, "r", encoding="utf-8") as f:
    explanation_text = f.read()


display(Markdown(explanation_text))

---